# PUBG Winning Factors: PySpark Feature Engineering + Machine Learning Pipeline

## Research question

**What in-game behaviors explain high PUBG placement, and how do these factors differ across solo, duo, and squad modes?**


## Modeling unit by mode

| Mode | Regression unit | Classification unit | Clustering unit |
|---|---|---|---|
| solo | one player row | one player row | one player row |
| duo | one team row = `matchId + groupId` | one team row | full duo teams only |
| squad | one team row = `matchId + groupId` | one team row | full squad teams only |

## Important implementation note
The main non-linear model is implemented with Spark MLlib's gradient-boosted trees:

- `GBTRegressor` for regression;
- `GBTClassifier` for Top 25% classification.

The core modeling idea: **linear/logistic baseline + non-linear boosting tree model**.  


---
# 0. Start Spark session and import packages

This section follows the same setup pattern as the Spark lecture notebooks.

If PySpark is not installed in your environment, uncomment the install line.

In [ ]:
# If your environment does not have pyspark installed, uncomment the next line.
# !pip install pyspark

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark import StorageLevel

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator

%matplotlib inline

# Local PySpark can easily run out of memory when many tasks run at once.
# local[2] is slower than local[*], but it is more stable on laptops.
# The rest of the notebook keeps the same analysis logic.
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("PUBG_PySpark_ML")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.driver.memory", "6g")
    .config("spark.memory.fraction", "0.6")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.shuffle.partitions", "32")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

print("Spark version:", spark.version)

# 1. Configuration

`USE_SAMPLE_FOR_DEVELOPMENT` is only for quick testing. For final results, keep it as `False`.

In [ ]:
DATA_PATH = "pubg_main_final.csv"
OUTPUT_DIR = "pubg_spark_outputs"

RANDOM_STATE = 42
TOP25_THRESHOLD = 0.75
TEST_FRACTION = 0.20
EPS = 1e-6 # avoid dividing by 0

USE_SAMPLE_FOR_DEVELOPMENT = False
SAMPLE_FRACTION = 0.10

# On a local laptop, exact diagnostic counts plus cache() can crash Spark.
# Keep this False for the full dataset. It only affects printed diagnostics, not modeling data.
RUN_HEAVY_DIAGNOSTIC_COUNTS = False

# GBT parameters. These are intentionally moderate so the notebook can run on a large dataset.
GBT_MAX_ITER = 40
GBT_MAX_DEPTH = 5

# Final K for clustering. We still inspect silhouette scores before using these values.
SOLO_K = 5
DUO_K = 5
SQUAD_K = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Load cleaned data


In [ ]:
myData = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

print("Number of columns:", len(myData.columns))
myData.printSchema()
myData.show(5)

In [ ]:
# Create mode and perspective from matchType.
# This repeats the earlier cleaning logic so the modeling notebook is complete.

myData = (
    myData
    .dropna(subset=["winPlacePerc"]) #Delete the rows with missing values for the target variable
    #.withColumn("matchType_lower", f.lower(f.col("matchType")))
    .withColumn(
        "mode",
        f.when(f.col("matchType").contains("solo"), f.lit("solo"))
         .when(f.col("matchType").contains("duo"), f.lit("duo"))
         .when(f.col("matchType").contains("squad"), f.lit("squad"))
         .otherwise(f.lit("other"))
    )
    .withColumn(
        "perspective",
        f.when(f.col("matchType").contains("fpp"), f.lit("fpp")).otherwise(f.lit("tpp"))
    )
    .withColumn(
        "expectedTeamSize",
        f.when(f.col("mode") == "solo", f.lit(1))
         .when(f.col("mode") == "duo", f.lit(2))
         .when(f.col("mode") == "squad", f.lit(4))
         .otherwise(f.lit(None))
    )
    .filter(f.col("mode").isin("solo", "duo", "squad"))
    .withColumn("perspectiveIndex", f.when(f.col("perspective") == "fpp", f.lit(1.0)).otherwise(f.lit(0.0)))
)

myData.select("Id", "matchId", "matchType", "mode", "perspective", "expectedTeamSize", "winPlacePerc").show(5)

# 3. Basic checks for duo/squad group consistency

We use `matchId + groupId` as the team identifier for duo and squad analysis.

In a team, all of the group members should share the same winPlacePerc, and we must check it for every group. The exact check is now performed later inside the team-level dataset construction by storing the minimum and maximum `winPlacePerc` for each team. And we will judge wether the group is satisfied by whether target_max - target_min is zero.

# 4. Player-level feature engineering

These features are created for every player row first.  
Solo modeling uses these features directly. Duo and squad modeling aggregate them into team-level features.

## Feature groups

**Mobility**

- `totalDistance`——衡量这个玩家有没有在“参与游戏”；EDA已发现移动距离和winPlacePerc呈正相关，背后反映：高移动 = 更好的位置选择+更少被毒圈淘汰；
- `walkRatio`—— 衡量移动方式：是“靠走”还是“靠车”；
- `rideRatio`—— 衡量是否依赖载具进行战略移动
- `swimRatio`——捕捉极端策略行为，用于识别苟分玩家

**Combat efficiency**

- `damagePerKill`——捕捉击杀效率
- `headshotRate`——衡量是否为精准性高玩
- `killEfficiency`—— kills / (weaponsAcquired + 1)，因为有些玩家疯狂捡装备但不打架。高效率：少资源 → 多击杀

**Resource management**

- `healsAndBoosts`
- `boostHealRatio`——boostHealRatio = boosts / (heals + boost + eps)
- `hasBoosts`
- `hasHeals`

**Support and contribution**

- `supportActions`——supportActions = assists + revives，衡量是否为团队型玩家
- `combatContribution`—— combatContribution = kills + assists + DBNOs，衡量在“战斗中”的总参与度

In [ ]:
myData = (
    myData
    # Mobility features
    .withColumn("totalDistance", f.col("walkDistance") + f.col("rideDistance") + f.col("swimDistance"))
    .withColumn("walkRatio", f.col("walkDistance") / (f.col("totalDistance") + f.lit(EPS)))
    .withColumn("rideRatio", f.col("rideDistance") / (f.col("totalDistance") + f.lit(EPS)))
    .withColumn("swimRatio", f.col("swimDistance") / (f.col("totalDistance") + f.lit(EPS)))

    # Combat efficiency features
    .withColumn("damagePerKill", f.col("damageDealt") / (f.col("kills") + f.lit(1.0)))
    .withColumn("headshotRate", f.col("headshotKills") / (f.col("kills") + f.lit(1.0)))
    .withColumn("killEfficiency", f.col("kills") / (f.col("weaponsAcquired") + f.lit(1.0)))

    # Resource management features
    .withColumn("healsAndBoosts", f.col("heals") + f.col("boosts"))
    .withColumn("boostHealRatio", f.col("boosts") / (f.col("heals") + f.col("boosts") + f.lit(EPS)))
    .withColumn("hasBoosts", f.when(f.col("boosts") > 0, f.lit(1)).otherwise(f.lit(0)))
    .withColumn("hasHeals", f.when(f.col("heals") > 0, f.lit(1)).otherwise(f.lit(0)))

    # Team support and general contribution features
    .withColumn("supportActions", f.col("assists") + f.col("revives"))
    .withColumn("combatContribution", f.col("kills") + f.col("assists") + f.col("DBNOs"))

    # Classification label
    .withColumn("isTop25", f.when(f.col("winPlacePerc") >= TOP25_THRESHOLD, f.lit(1)).otherwise(f.lit(0)))
)

# Replace possible missing numeric values with 0 after feature engineering.
myData = myData.fillna(0)
myData
myData.select(
    "mode", "kills", "damageDealt", "boosts", "heals", "walkDistance", "rideDistance",
    "totalDistance", "damagePerKill", "healsAndBoosts", "boostHealRatio", "isTop25"
).show(10)

# 5. Build solo player data

Solo outcomes are individual outcomes, so one row is one player.

In [ ]:
soloData = myData.filter(f.col("mode") == "solo")

soloData.select("Id", "matchId", "kills", "damageDealt", "walkDistance", "boosts", "winPlacePerc", "isTop25").show(5)


# 6. Build duo and squad team-level data

For duo and squad, the final placement belongs to a team.  
Therefore, we group by:

```python
matchId + groupId
```

## Handling different team sizes

Some duo/squad teams may have fewer than the expected number of players. This is not automatically treated as an error. Instead, we keep the team and create:

- `groupSize`
- `fillRatio`—— fillRatio = groupSize / expectedTeamSize
- `isFullTeam`

This allows the model to learn whether underfilled teams perform differently.

In [ ]:
# Columns to aggregate at team level.

team_sum_cols = [
    "assists", "boosts", "damageDealt", "DBNOs", "headshotKills", "heals",
    "kills", "killStreaks", "revives", "rideDistance", "roadKills", "swimDistance",
    "teamKills", "vehicleDestroys", "walkDistance", "weaponsAcquired",
    "totalDistance", "healsAndBoosts", "supportActions", "combatContribution"
]

team_mean_cols = [
    "assists", "boosts", "damageDealt", "DBNOs", "headshotKills", "heals",
    "kills", "killStreaks", "longestKill", "revives", "rideDistance", "roadKills",
    "swimDistance", "teamKills", "vehicleDestroys", "walkDistance", "weaponsAcquired",
    "totalDistance", "damagePerKill", "headshotRate", "killEfficiency",
    "healsAndBoosts", "boostHealRatio", "supportActions", "combatContribution"
]

# 因为max问的是队伍里某个成员在某项具体行为上是不是特别突出？所以更适合将 原始的、单一行为的 变量放入，故“supportActions”这一类我就不放进去了 
team_max_cols = [ # we use 'max' to catch whether there is a particularly strong carry player in the team?
    "kills", "damageDealt", "DBNOs", "boosts", "heals", "walkDistance", "rideDistance",
    "swimDistance", "totalDistance", "weaponsAcquired", "assists", "revives",
    "longestKill", "headshotKills", "killStreaks"
]

team_std_cols = [ # 在均衡的队伍里，对应的std可能很小；但是在有强者带飞的队伍，可能对应的std值很大。
    "kills", "damageDealt","DBNOs" , "boosts", "heals", "walkDistance", "rideDistance",
    "totalDistance", "weaponsAcquired", "assists", "revives"
]

In [ ]:
# Build the aggregation expressions for team rows.
# This is still Spark DataFrame style: groupBy + agg.

team_agg_exprs = [
    f.count("*").alias("groupSize"),
    # For the same team, these variables are theoretically the same for all players within the team, 
    # so the first one is taken directly.
    f.first("mode").alias("mode"),
    f.first("perspectiveIndex").alias("perspectiveIndex"),
    f.first("expectedTeamSize").alias("expectedTeamSize"),
    f.first("matchDuration").alias("matchDuration"),
    f.first("maxPlace").alias("maxPlace"),
    f.first("numGroups").alias("numGroups"),
    f.first("winPlacePerc").alias("winPlacePerc"),
    f.min("winPlacePerc").alias("target_min"),
    f.max("winPlacePerc").alias("target_max")
]

for c in team_sum_cols:
    team_agg_exprs.append(f.sum(c).alias(c + "_sum"))

for c in team_mean_cols:
    team_agg_exprs.append(f.avg(c).alias(c + "_mean"))

for c in team_max_cols:
    team_agg_exprs.append(f.max(c).alias(c + "_max"))

for c in team_std_cols:
    team_agg_exprs.append(f.stddev_samp(c).alias(c + "_std"))

In [ ]:
# Create duo team data.
duoTeamData = (
    myData
    .filter(f.col("mode") == "duo")
    .groupBy("matchId", "groupId")
    .agg(*team_agg_exprs)
    .fillna(0)
    .withColumn("fillRatio", f.col("groupSize") / f.col("expectedTeamSize"))
    .withColumn("isFullTeam", f.when(f.col("groupSize") == f.col("expectedTeamSize"), f.lit(1)).otherwise(f.lit(0)))
    .withColumn("targetRange", f.abs(f.col("target_max") - f.col("target_min")))
    .withColumn("isTop25", f.when(f.col("winPlacePerc") >= TOP25_THRESHOLD, f.lit(1)).otherwise(f.lit(0)))
    .withColumn("topKillShare", f.col("kills_max") / (f.col("kills_sum") + f.lit(EPS)))
    .withColumn("topDamageShare", f.col("damageDealt_max") / (f.col("damageDealt_sum") + f.lit(EPS)))
    .withColumn("topBoostShare", f.col("boosts_max") / (f.col("boosts_sum") + f.lit(EPS)))
    .withColumn("topWalkShare", f.col("walkDistance_max") / (f.col("walkDistance_sum") + f.lit(EPS)))
)

# Show examples only. Exact team counts are computed later in the summary section if needed.
duoTeamData.select("matchId", "groupId", "groupSize", "fillRatio", "isFullTeam", "kills_sum", "damageDealt_sum", "boosts_sum", "winPlacePerc").show(5)

In [ ]:
# Create squad team data.
squadTeamData = (
    myData
    .filter(f.col("mode") == "squad")
    .groupBy("matchId", "groupId")
    .agg(*team_agg_exprs)
    .fillna(0)
    .withColumn("fillRatio", f.col("groupSize") / f.col("expectedTeamSize"))
    .withColumn("isFullTeam", f.when(f.col("groupSize") == f.col("expectedTeamSize"), f.lit(1)).otherwise(f.lit(0)))
    .withColumn("targetRange", f.abs(f.col("target_max") - f.col("target_min")))
    .withColumn("isTop25", f.when(f.col("winPlacePerc") >= TOP25_THRESHOLD, f.lit(1)).otherwise(f.lit(0)))
    # 这里的 top-Share feature 衡量的是队伍中表现最突出的那个人占全队总贡献的比例
    .withColumn("topKillShare", f.col("kills_max") / (f.col("kills_sum") + f.lit(EPS)))
    .withColumn("topDamageShare", f.col("damageDealt_max") / (f.col("damageDealt_sum") + f.lit(EPS)))
    .withColumn("topBoostShare", f.col("boosts_max") / (f.col("boosts_sum") + f.lit(EPS)))
    .withColumn("topWalkShare", f.col("walkDistance_max") / (f.col("walkDistance_sum") + f.lit(EPS)))
)

# Show examples only. Exact team counts are computed later in the summary section if needed.
squadTeamData.select("matchId", "groupId", "groupSize", "fillRatio", "isFullTeam", "kills_sum", "damageDealt_sum", "boosts_sum", "winPlacePerc").show(5)

# 8. Feature lists for modeling

We do **not** use ID columns such as `Id`, `groupId`, or `matchId` as predictors.

We also do **not** include `killPlace`, `rankPoints`, `killPoints`, or `winPoints` in the main model.  
The purpose here is to explain current-match behavior rather than use ranking-like or historical rating variables.

In [ ]:
solo_features = [
    "assists", "boosts", "damageDealt", "DBNOs", "headshotKills", "heals", 
    "kills", "killStreaks", "longestKill",  "maxPlace", 
    "numGroups", "revives", "rideDistance", "roadKills", "swimDistance", 
    "teamKills", "vehicleDestroys", "walkDistance", "weaponsAcquired", 
    "perspectiveIndex", "totalDistance", "walkRatio", "rideRatio", "swimRatio",
    "damagePerKill", "headshotRate", "killEfficiency", "healsAndBoosts", 
    "boostHealRatio", "hasBoosts", "hasHeals", "supportActions", "combatContribution",
]

team_basic_features = [# 主要是"groupSize" + "fillRatio" + "isFullTeam" + sum feature（只看队伍总体产出）
    "groupSize", "fillRatio", "isFullTeam", "perspectiveIndex", "maxPlace", "numGroups",
    "assists_sum", "boosts_sum", "damageDealt_sum", "DBNOs_sum", "headshotKills_sum",
    "heals_sum", "kills_sum", "killStreaks_sum", "revives_sum", "rideDistance_sum",
    "roadKills_sum", "swimDistance_sum", "teamKills_sum", "vehicleDestroys_sum",
    "walkDistance_sum", "weaponsAcquired_sum", "totalDistance_sum", "healsAndBoosts_sum",
    "supportActions_sum", "combatContribution_sum"
]

team_full_features = team_basic_features + [# 加入了mean features, max features, std features, top-share features
    "assists_mean", "boosts_mean", "damageDealt_mean", "DBNOs_mean", "headshotKills_mean",
    "heals_mean", "kills_mean", "killStreaks_mean", "longestKill_mean", "revives_mean",
    "rideDistance_mean", "roadKills_mean", "swimDistance_mean", "teamKills_mean",
    "vehicleDestroys_mean", "walkDistance_mean", "weaponsAcquired_mean", "totalDistance_mean",
    "damagePerKill_mean", "headshotRate_mean", "killEfficiency_mean", "healsAndBoosts_mean",
    "boostHealRatio_mean", "supportActions_mean", "combatContribution_mean",
    "kills_max", "damageDealt_max", "DBNOs_max", "boosts_max", "heals_max",
    "walkDistance_max", "rideDistance_max", "swimDistance_max", "totalDistance_max",
    "weaponsAcquired_max", "assists_max", "revives_max", "longestKill_max",
    "headshotKills_max", "killStreaks_max",
    "kills_std", "damageDealt_std", "DBNOs_std", "boosts_std", "heals_std", "walkDistance_std",
    "rideDistance_std", "totalDistance_std", "weaponsAcquired_std", "assists_std", "revives_std", 
    "topKillShare", "topDamageShare", "topBoostShare", "topWalkShare"
]

print("Number of solo features:", len(solo_features))
print("Number of basic team features:", len(team_basic_features))
print("Number of full team-composition features:", len(team_full_features))

# 9. Train-test split by matchId

A normal row-level random split can leak information because players or teams from the same match may appear in both training and test sets.

To reduce this leakage, the split is done at the `matchId` level:

1. get distinct `matchId` values;
2. split the match IDs;
3. join the selected match IDs back to the modeling data.

In [ ]:
# Solo split by matchId.
soloMatches = soloData.select("matchId").distinct()
soloTrainMatches, soloTestMatches = soloMatches.randomSplit([1 - TEST_FRACTION, TEST_FRACTION], seed=RANDOM_STATE)

soloTrain = soloData.join(soloTrainMatches, on="matchId", how="inner")
soloTest = soloData.join(soloTestMatches, on="matchId", how="inner")



In [ ]:
# Duo split by matchId.
duoMatches = duoTeamData.select("matchId").distinct()
duoTrainMatches, duoTestMatches = duoMatches.randomSplit([1 - TEST_FRACTION, TEST_FRACTION], seed=RANDOM_STATE)

duoTrain = duoTeamData.join(duoTrainMatches, on="matchId", how="inner")
duoTest = duoTeamData.join(duoTestMatches, on="matchId", how="inner")



In [ ]:
# Squad split by matchId.
squadMatches = squadTeamData.select("matchId").distinct()
squadTrainMatches, squadTestMatches = squadMatches.randomSplit([1 - TEST_FRACTION, TEST_FRACTION], seed=RANDOM_STATE)

squadTrain = squadTeamData.join(squadTrainMatches, on="matchId", how="inner")
squadTest = squadTeamData.join(squadTestMatches, on="matchId", how="inner")



---
# 10. Regression: predicting `winPlacePerc`

Regression answers the question:

> Can we predict the final placement percentile from post-match behavior statistics?

Models used:

1. **Linear Regression baseline**: baseline.
2. **Gradient-Boosted Trees Regressor**: non-linear boosting model.

Evaluation metrics:

- **MAE**: average absolute error in placement percentile;
- **RMSE**: penalizes larger errors;
- **R²**: proportion of variation explained by the model.

In [ ]:
regression_results = []

reg_eval_mae = RegressionEvaluator(labelCol="winPlacePerc", predictionCol="prediction", metricName="mae")
reg_eval_rmse = RegressionEvaluator(labelCol="winPlacePerc", predictionCol="prediction", metricName="rmse")
reg_eval_r2 = RegressionEvaluator(labelCol="winPlacePerc", predictionCol="prediction", metricName="r2")

## 10.1 Solo regression

Unit: one solo player row.

In [ ]:
# Linear Regression baseline for solo.
solo_lr_assembler = VectorAssembler(inputCols=solo_features, outputCol="features", handleInvalid="keep")
solo_lr_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
solo_lr = LinearRegression(labelCol="winPlacePerc", featuresCol="scaledFeatures", maxIter=50)

solo_lr_pipeline = Pipeline(stages=[solo_lr_assembler, solo_lr_scaler, solo_lr])
solo_lr_model = solo_lr_pipeline.fit(soloTrain)
solo_lr_pred = solo_lr_model.transform(soloTest)

solo_lr_mae = reg_eval_mae.evaluate(solo_lr_pred)
solo_lr_rmse = reg_eval_rmse.evaluate(solo_lr_pred)
solo_lr_r2 = reg_eval_r2.evaluate(solo_lr_pred)

regression_results.append(("solo", "player", "Linear Regression baseline", solo_lr_mae, solo_lr_rmse, solo_lr_r2))

print("Solo Linear Regression")
print("MAE:", solo_lr_mae)
print("RMSE:", solo_lr_rmse)
print("R2:", solo_lr_r2)
solo_lr_pred.select("prediction", "winPlacePerc").show(10)

In [ ]:
# GBT Regressor for solo.
solo_gbt_assembler = VectorAssembler(inputCols=solo_features, outputCol="features", handleInvalid="keep")
solo_gbt = GBTRegressor(
    labelCol="winPlacePerc",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

solo_gbt_pipeline = Pipeline(stages=[solo_gbt_assembler, solo_gbt])
solo_gbt_model = solo_gbt_pipeline.fit(soloTrain)
solo_gbt_pred = solo_gbt_model.transform(soloTest)

solo_gbt_mae = reg_eval_mae.evaluate(solo_gbt_pred)
solo_gbt_rmse = reg_eval_rmse.evaluate(solo_gbt_pred)
solo_gbt_r2 = reg_eval_r2.evaluate(solo_gbt_pred)

regression_results.append(("solo", "player", "GBTRegressor", solo_gbt_mae, solo_gbt_rmse, solo_gbt_r2))

print("Solo GBT Regressor")
print("MAE:", solo_gbt_mae)
print("RMSE:", solo_gbt_rmse)
print("R2:", solo_gbt_r2)
solo_gbt_pred.select("prediction", "winPlacePerc").show(10)

## 10.2 Duo regression

Unit: one duo team row.

We run two team-level versions:

1. **basic team model**: team totals and basic team-size features;
2. **full team-composition model**: basic features plus means, maximums, standard deviations, and top-player contribution shares.

In [ ]:
# Duo basic team model: Linear Regression baseline.
duo_basic_lr_assembler = VectorAssembler(inputCols=team_basic_features, outputCol="features", handleInvalid="keep")
duo_basic_lr_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
duo_basic_lr = LinearRegression(labelCol="winPlacePerc", featuresCol="scaledFeatures", maxIter=50)

duo_basic_lr_pipeline = Pipeline(stages=[duo_basic_lr_assembler, duo_basic_lr_scaler, duo_basic_lr])
duo_basic_lr_model = duo_basic_lr_pipeline.fit(duoTrain)
duo_basic_lr_pred = duo_basic_lr_model.transform(duoTest)

duo_basic_lr_mae = reg_eval_mae.evaluate(duo_basic_lr_pred)
duo_basic_lr_rmse = reg_eval_rmse.evaluate(duo_basic_lr_pred)
duo_basic_lr_r2 = reg_eval_r2.evaluate(duo_basic_lr_pred)

regression_results.append(("duo", "team_basic", "Linear Regression baseline", duo_basic_lr_mae, duo_basic_lr_rmse, duo_basic_lr_r2))

print("Duo basic Linear Regression")
print("MAE:", duo_basic_lr_mae)
print("RMSE:", duo_basic_lr_rmse)
print("R2:", duo_basic_lr_r2)

In [ ]:
# Duo basic team model: GBT Regressor.
duo_basic_gbt_assembler = VectorAssembler(inputCols=team_basic_features, outputCol="features", handleInvalid="keep")
duo_basic_gbt = GBTRegressor(
    labelCol="winPlacePerc",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

duo_basic_gbt_pipeline = Pipeline(stages=[duo_basic_gbt_assembler, duo_basic_gbt])
duo_basic_gbt_model = duo_basic_gbt_pipeline.fit(duoTrain)
duo_basic_gbt_pred = duo_basic_gbt_model.transform(duoTest)

duo_basic_gbt_mae = reg_eval_mae.evaluate(duo_basic_gbt_pred)
duo_basic_gbt_rmse = reg_eval_rmse.evaluate(duo_basic_gbt_pred)
duo_basic_gbt_r2 = reg_eval_r2.evaluate(duo_basic_gbt_pred)

regression_results.append(("duo", "team_basic", "GBTRegressor", duo_basic_gbt_mae, duo_basic_gbt_rmse, duo_basic_gbt_r2))

print("Duo basic GBT Regressor")
print("MAE:", duo_basic_gbt_mae)
print("RMSE:", duo_basic_gbt_rmse)
print("R2:", duo_basic_gbt_r2)

In [ ]:
# Duo full team-composition model: Linear Regression baseline.
duo_full_lr_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
duo_full_lr_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
duo_full_lr = LinearRegression(labelCol="winPlacePerc", featuresCol="scaledFeatures", maxIter=50)

duo_full_lr_pipeline = Pipeline(stages=[duo_full_lr_assembler, duo_full_lr_scaler, duo_full_lr])
duo_full_lr_model = duo_full_lr_pipeline.fit(duoTrain)
duo_full_lr_pred = duo_full_lr_model.transform(duoTest)

duo_full_lr_mae = reg_eval_mae.evaluate(duo_full_lr_pred)
duo_full_lr_rmse = reg_eval_rmse.evaluate(duo_full_lr_pred)
duo_full_lr_r2 = reg_eval_r2.evaluate(duo_full_lr_pred)

regression_results.append(("duo", "team_full", "Linear Regression baseline", duo_full_lr_mae, duo_full_lr_rmse, duo_full_lr_r2))

print("Duo full Linear Regression")
print("MAE:", duo_full_lr_mae)
print("RMSE:", duo_full_lr_rmse)
print("R2:", duo_full_lr_r2)

In [ ]:
# Duo full team-composition model: GBT Regressor.
duo_full_gbt_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
duo_full_gbt = GBTRegressor(
    labelCol="winPlacePerc",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

duo_full_gbt_pipeline = Pipeline(stages=[duo_full_gbt_assembler, duo_full_gbt])
duo_full_gbt_model = duo_full_gbt_pipeline.fit(duoTrain)
duo_full_gbt_pred = duo_full_gbt_model.transform(duoTest)

duo_full_gbt_mae = reg_eval_mae.evaluate(duo_full_gbt_pred)
duo_full_gbt_rmse = reg_eval_rmse.evaluate(duo_full_gbt_pred)
duo_full_gbt_r2 = reg_eval_r2.evaluate(duo_full_gbt_pred)

regression_results.append(("duo", "team_full", "GBTRegressor", duo_full_gbt_mae, duo_full_gbt_rmse, duo_full_gbt_r2))

print("Duo full GBT Regressor")
print("MAE:", duo_full_gbt_mae)
print("RMSE:", duo_full_gbt_rmse)
print("R2:", duo_full_gbt_r2)

## 10.3 Squad regression

Unit: one squad team row.

As with duo, squad has a basic team model and a full team-composition model.

In [ ]:
# Squad basic team model: Linear Regression baseline.
squad_basic_lr_assembler = VectorAssembler(inputCols=team_basic_features, outputCol="features", handleInvalid="keep")
squad_basic_lr_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
squad_basic_lr = LinearRegression(labelCol="winPlacePerc", featuresCol="scaledFeatures", maxIter=50)

squad_basic_lr_pipeline = Pipeline(stages=[squad_basic_lr_assembler, squad_basic_lr_scaler, squad_basic_lr])
squad_basic_lr_model = squad_basic_lr_pipeline.fit(squadTrain)
squad_basic_lr_pred = squad_basic_lr_model.transform(squadTest)

squad_basic_lr_mae = reg_eval_mae.evaluate(squad_basic_lr_pred)
squad_basic_lr_rmse = reg_eval_rmse.evaluate(squad_basic_lr_pred)
squad_basic_lr_r2 = reg_eval_r2.evaluate(squad_basic_lr_pred)

regression_results.append(("squad", "team_basic", "Linear Regression baseline", squad_basic_lr_mae, squad_basic_lr_rmse, squad_basic_lr_r2))

print("Squad basic Linear Regression")
print("MAE:", squad_basic_lr_mae)
print("RMSE:", squad_basic_lr_rmse)
print("R2:", squad_basic_lr_r2)

In [ ]:
# Squad basic team model: GBT Regressor.
squad_basic_gbt_assembler = VectorAssembler(inputCols=team_basic_features, outputCol="features", handleInvalid="keep")
squad_basic_gbt = GBTRegressor(
    labelCol="winPlacePerc",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

squad_basic_gbt_pipeline = Pipeline(stages=[squad_basic_gbt_assembler, squad_basic_gbt])
squad_basic_gbt_model = squad_basic_gbt_pipeline.fit(squadTrain)
squad_basic_gbt_pred = squad_basic_gbt_model.transform(squadTest)

squad_basic_gbt_mae = reg_eval_mae.evaluate(squad_basic_gbt_pred)
squad_basic_gbt_rmse = reg_eval_rmse.evaluate(squad_basic_gbt_pred)
squad_basic_gbt_r2 = reg_eval_r2.evaluate(squad_basic_gbt_pred)

regression_results.append(("squad", "team_basic", "GBTRegressor", squad_basic_gbt_mae, squad_basic_gbt_rmse, squad_basic_gbt_r2))

print("Squad basic GBT Regressor")
print("MAE:", squad_basic_gbt_mae)
print("RMSE:", squad_basic_gbt_rmse)
print("R2:", squad_basic_gbt_r2)

In [ ]:
# Squad full team-composition model: Linear Regression baseline.
squad_full_lr_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
squad_full_lr_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
squad_full_lr = LinearRegression(labelCol="winPlacePerc", featuresCol="scaledFeatures", maxIter=50)

squad_full_lr_pipeline = Pipeline(stages=[squad_full_lr_assembler, squad_full_lr_scaler, squad_full_lr])
squad_full_lr_model = squad_full_lr_pipeline.fit(squadTrain)
squad_full_lr_pred = squad_full_lr_model.transform(squadTest)

squad_full_lr_mae = reg_eval_mae.evaluate(squad_full_lr_pred)
squad_full_lr_rmse = reg_eval_rmse.evaluate(squad_full_lr_pred)
squad_full_lr_r2 = reg_eval_r2.evaluate(squad_full_lr_pred)

regression_results.append(("squad", "team_full", "Linear Regression baseline", squad_full_lr_mae, squad_full_lr_rmse, squad_full_lr_r2))

print("Squad full Linear Regression")
print("MAE:", squad_full_lr_mae)
print("RMSE:", squad_full_lr_rmse)
print("R2:", squad_full_lr_r2)

In [ ]:
# Squad full team-composition model: GBT Regressor.
squad_full_gbt_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
squad_full_gbt = GBTRegressor(
    labelCol="winPlacePerc",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

squad_full_gbt_pipeline = Pipeline(stages=[squad_full_gbt_assembler, squad_full_gbt])
squad_full_gbt_model = squad_full_gbt_pipeline.fit(squadTrain)
squad_full_gbt_pred = squad_full_gbt_model.transform(squadTest)

squad_full_gbt_mae = reg_eval_mae.evaluate(squad_full_gbt_pred)
squad_full_gbt_rmse = reg_eval_rmse.evaluate(squad_full_gbt_pred)
squad_full_gbt_r2 = reg_eval_r2.evaluate(squad_full_gbt_pred)

regression_results.append(("squad", "team_full", "GBTRegressor", squad_full_gbt_mae, squad_full_gbt_rmse, squad_full_gbt_r2))

print("Squad full GBT Regressor")
print("MAE:", squad_full_gbt_mae)
print("RMSE:", squad_full_gbt_rmse)
print("R2:", squad_full_gbt_r2)

## 10.4 Regression result table

Use this table in the report to compare:

- linear baseline vs boosting model;
- duo/squad basic team model vs full team-composition model.

In [ ]:
# The model metrics are already stored in a small Python list.
# We use pandas here instead of spark.createDataFrame because creating a Spark DataFrame
# from local Python objects may start a Python worker. On some Windows/local Spark setups,
# that can fail after many previous Spark stages even though the metrics are tiny.
# This does not change any model result or any later modeling dataset.

regressionResults = pd.DataFrame(
    regression_results,
    columns=["mode", "unit_or_version", "model", "MAE", "RMSE", "R2"]
)

regressionResults = regressionResults.sort_values(["mode", "unit_or_version", "model"])
print(regressionResults.to_string(index=False))
regressionResults.to_csv(os.path.join(OUTPUT_DIR, "regression_results.csv"), index=False)


## 10.5 Team regression comparison

This table directly compares the basic team model and the full team-composition model for duo and squad.

The purpose is to check whether team size, average/max contribution, within-team variation, and carry-share features add explanatory power beyond raw team totals.

In [ ]:
# Compare basic team GBT and full team-composition GBT for duo and squad.
# This is a small pandas table built from regressionResults, so it does not create a Spark job.

team_comparison_rows = []

for mode_name in ["duo", "squad"]:
    temp = regressionResults[(regressionResults["mode"] == mode_name) & (regressionResults["model"] == "GBTRegressor")].copy()
    temp = temp.set_index("unit_or_version")

    if "team_basic" in temp.index and "team_full" in temp.index:
        basic = temp.loc["team_basic"]
        full = temp.loc["team_full"]

        team_comparison_rows.append({
            "mode": mode_name,
            "basic_MAE": basic["MAE"],
            "full_MAE": full["MAE"],
            "MAE_improvement": basic["MAE"] - full["MAE"],
            "basic_RMSE": basic["RMSE"],
            "full_RMSE": full["RMSE"],
            "RMSE_improvement": basic["RMSE"] - full["RMSE"],
            "basic_R2": basic["R2"],
            "full_R2": full["R2"],
            "R2_improvement": full["R2"] - basic["R2"]
        })

teamRegressionComparison = pd.DataFrame(team_comparison_rows)
print(teamRegressionComparison.to_string(index=False))
teamRegressionComparison.to_csv(os.path.join(OUTPUT_DIR, "team_regression_comparison.csv"), index=False)


# 11. Feature importance from GBT regression models

Tree-based models provide feature importance values.  
These values help us interpret which behavior variables were most useful for predicting placement.

For duo and squad, the most important comparison is the **full team-composition model**, because it includes team size, means, maximums, standard deviations, and contribution-share variables.

In [ ]:
# Solo GBT feature importance.
# Feature importance values are already collected from the trained Spark model object.
# The final table is small, so pandas is safer than spark.createDataFrame here.

solo_gbt_importance_values = solo_gbt_model.stages[-1].featureImportances.toArray().tolist()
solo_gbt_importance_rows = []

for i in range(len(solo_features)):
    solo_gbt_importance_rows.append((solo_features[i], float(solo_gbt_importance_values[i])))

soloGbtImportance = pd.DataFrame(solo_gbt_importance_rows, columns=["feature", "importance"])
soloGbtImportance = soloGbtImportance.sort_values("importance", ascending=False)
print(soloGbtImportance.head(20).to_string(index=False))
soloGbtImportance.to_csv(os.path.join(OUTPUT_DIR, "solo_gbt_regression_feature_importance.csv"), index=False)


In [ ]:
# Duo full team GBT feature importance.
# This is a small local table created from the trained model's featureImportances vector.

duo_full_gbt_importance_values = duo_full_gbt_model.stages[-1].featureImportances.toArray().tolist()
duo_full_gbt_importance_rows = []

for i in range(len(team_full_features)):
    duo_full_gbt_importance_rows.append((team_full_features[i], float(duo_full_gbt_importance_values[i])))

duoFullGbtImportance = pd.DataFrame(duo_full_gbt_importance_rows, columns=["feature", "importance"])
duoFullGbtImportance = duoFullGbtImportance.sort_values("importance", ascending=False)
print(duoFullGbtImportance.head(20).to_string(index=False))
duoFullGbtImportance.to_csv(os.path.join(OUTPUT_DIR, "duo_full_gbt_regression_feature_importance.csv"), index=False)


In [ ]:
# Squad full team GBT feature importance.
# This is a small local table created from the trained model's featureImportances vector.

squad_full_gbt_importance_values = squad_full_gbt_model.stages[-1].featureImportances.toArray().tolist()
squad_full_gbt_importance_rows = []

for i in range(len(team_full_features)):
    squad_full_gbt_importance_rows.append((team_full_features[i], float(squad_full_gbt_importance_values[i])))

squadFullGbtImportance = pd.DataFrame(squad_full_gbt_importance_rows, columns=["feature", "importance"])
squadFullGbtImportance = squadFullGbtImportance.sort_values("importance", ascending=False)
print(squadFullGbtImportance.head(20).to_string(index=False))
squadFullGbtImportance.to_csv(os.path.join(OUTPUT_DIR, "squad_full_gbt_regression_feature_importance.csv"), index=False)


---
# 12. Classification: predicting Top 25% placement

Classification answers a different question from regression:

> Can we identify whether a player/team will finish in the Top 25%?

Why Top 25%?

- It still represents a clearly successful outcome.
- It is less class-imbalanced than Top 10%.
- It works consistently across solo, duo, and squad.

Models used:

1. **Logistic Regression baseline**;
2. **GBTClassifier** as the non-linear boosting model.

In [ ]:
classification_results = []

binary_eval_roc = BinaryClassificationEvaluator(labelCol="isTop25", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
binary_eval_pr = BinaryClassificationEvaluator(labelCol="isTop25", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
multiclass_eval_accuracy = MulticlassClassificationEvaluator(labelCol="isTop25", predictionCol="prediction", metricName="accuracy")

# F1 is computed manually from Top25 precision and Top25 recall below.
# This is important because Spark's metricName="f1" returns weighted F1,
# while this project needs positive-class F1 for isTop25 = 1.

## 12.1 Solo classification

In [ ]:
# Solo Logistic Regression.
solo_log_assembler = VectorAssembler(inputCols=solo_features, outputCol="features", handleInvalid="keep")
solo_log_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
solo_log = LogisticRegression(labelCol="isTop25", featuresCol="scaledFeatures", maxIter=50)

solo_log_pipeline = Pipeline(stages=[solo_log_assembler, solo_log_scaler, solo_log])
solo_log_model = solo_log_pipeline.fit(soloTrain)
solo_log_pred = solo_log_model.transform(soloTest)

solo_log_accuracy = multiclass_eval_accuracy.evaluate(solo_log_pred)
solo_log_roc = binary_eval_roc.evaluate(solo_log_pred)
solo_log_pr = binary_eval_pr.evaluate(solo_log_pred)

# Compute Top25 precision, recall, and positive-class F1 from one small aggregate job.
# This avoids Spark's weighted-F1 definition and keeps the metric aligned with sklearn f1_score(pos_label=1).
solo_log_confusion_counts = solo_log_pred.agg(
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
    f.sum(f.when((f.col("isTop25") == 0) & (f.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 0), 1).otherwise(0)).alias("fn")
).collect()[0]

solo_log_tp = solo_log_confusion_counts["tp"]
solo_log_fp = solo_log_confusion_counts["fp"]
solo_log_fn = solo_log_confusion_counts["fn"]

solo_log_precision = solo_log_tp / (solo_log_tp + solo_log_fp) if (solo_log_tp + solo_log_fp) > 0 else 0
solo_log_recall = solo_log_tp / (solo_log_tp + solo_log_fn) if (solo_log_tp + solo_log_fn) > 0 else 0
solo_log_f1 = 2 * solo_log_precision * solo_log_recall / (solo_log_precision + solo_log_recall) if (solo_log_precision + solo_log_recall) > 0 else 0

classification_results.append(("solo", "player", "Logistic Regression baseline", solo_log_accuracy, solo_log_precision, solo_log_recall, solo_log_f1, solo_log_roc, solo_log_pr))

print("Solo Logistic Regression classification")
print("Accuracy:", solo_log_accuracy)
print("Precision:", solo_log_precision)
print("Recall:", solo_log_recall)
print("F1:", solo_log_f1)
print("ROC-AUC:", solo_log_roc)
print("PR-AUC:", solo_log_pr)
solo_log_pred.groupBy("isTop25", "prediction").count().orderBy("isTop25", "prediction").show()

In [ ]:
# Solo GBT Classifier.
solo_cls_gbt_assembler = VectorAssembler(inputCols=solo_features, outputCol="features", handleInvalid="keep")
solo_cls_gbt = GBTClassifier(
    labelCol="isTop25",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

solo_cls_gbt_pipeline = Pipeline(stages=[solo_cls_gbt_assembler, solo_cls_gbt])
solo_cls_gbt_model = solo_cls_gbt_pipeline.fit(soloTrain)
solo_cls_gbt_pred = solo_cls_gbt_model.transform(soloTest)

solo_cls_gbt_accuracy = multiclass_eval_accuracy.evaluate(solo_cls_gbt_pred)
solo_cls_gbt_roc = binary_eval_roc.evaluate(solo_cls_gbt_pred)
solo_cls_gbt_pr = binary_eval_pr.evaluate(solo_cls_gbt_pred)

# Compute Top25 precision, recall, and positive-class F1 from one small aggregate job.
# This avoids Spark's weighted-F1 definition and keeps the metric aligned with sklearn f1_score(pos_label=1).
solo_cls_gbt_confusion_counts = solo_cls_gbt_pred.agg(
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
    f.sum(f.when((f.col("isTop25") == 0) & (f.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 0), 1).otherwise(0)).alias("fn")
).collect()[0]

solo_cls_gbt_tp = solo_cls_gbt_confusion_counts["tp"]
solo_cls_gbt_fp = solo_cls_gbt_confusion_counts["fp"]
solo_cls_gbt_fn = solo_cls_gbt_confusion_counts["fn"]

solo_cls_gbt_precision = solo_cls_gbt_tp / (solo_cls_gbt_tp + solo_cls_gbt_fp) if (solo_cls_gbt_tp + solo_cls_gbt_fp) > 0 else 0
solo_cls_gbt_recall = solo_cls_gbt_tp / (solo_cls_gbt_tp + solo_cls_gbt_fn) if (solo_cls_gbt_tp + solo_cls_gbt_fn) > 0 else 0
solo_cls_gbt_f1 = 2 * solo_cls_gbt_precision * solo_cls_gbt_recall / (solo_cls_gbt_precision + solo_cls_gbt_recall) if (solo_cls_gbt_precision + solo_cls_gbt_recall) > 0 else 0

classification_results.append(("solo", "player", "GBTClassifier", solo_cls_gbt_accuracy, solo_cls_gbt_precision, solo_cls_gbt_recall, solo_cls_gbt_f1, solo_cls_gbt_roc, solo_cls_gbt_pr))

print("Solo GBT classification")
print("Accuracy:", solo_cls_gbt_accuracy)
print("Precision:", solo_cls_gbt_precision)
print("Recall:", solo_cls_gbt_recall)
print("F1:", solo_cls_gbt_f1)
print("ROC-AUC:", solo_cls_gbt_roc)
print("PR-AUC:", solo_cls_gbt_pr)
solo_cls_gbt_pred.groupBy("isTop25", "prediction").count().orderBy("isTop25", "prediction").show()

## 12.2 Duo classification

For duo classification, we use the full team-composition feature set because the target is a team-level outcome.

In [ ]:
# Duo Logistic Regression with full team-composition features.
duo_log_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
duo_log_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
duo_log = LogisticRegression(labelCol="isTop25", featuresCol="scaledFeatures", maxIter=50)

duo_log_pipeline = Pipeline(stages=[duo_log_assembler, duo_log_scaler, duo_log])
duo_log_model = duo_log_pipeline.fit(duoTrain)
duo_log_pred = duo_log_model.transform(duoTest)

duo_log_accuracy = multiclass_eval_accuracy.evaluate(duo_log_pred)
duo_log_roc = binary_eval_roc.evaluate(duo_log_pred)
duo_log_pr = binary_eval_pr.evaluate(duo_log_pred)

# Compute Top25 precision, recall, and positive-class F1 from one small aggregate job.
# This avoids Spark's weighted-F1 definition and keeps the metric aligned with sklearn f1_score(pos_label=1).
duo_log_confusion_counts = duo_log_pred.agg(
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
    f.sum(f.when((f.col("isTop25") == 0) & (f.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 0), 1).otherwise(0)).alias("fn")
).collect()[0]

duo_log_tp = duo_log_confusion_counts["tp"]
duo_log_fp = duo_log_confusion_counts["fp"]
duo_log_fn = duo_log_confusion_counts["fn"]

duo_log_precision = duo_log_tp / (duo_log_tp + duo_log_fp) if (duo_log_tp + duo_log_fp) > 0 else 0
duo_log_recall = duo_log_tp / (duo_log_tp + duo_log_fn) if (duo_log_tp + duo_log_fn) > 0 else 0
duo_log_f1 = 2 * duo_log_precision * duo_log_recall / (duo_log_precision + duo_log_recall) if (duo_log_precision + duo_log_recall) > 0 else 0

classification_results.append(("duo", "team_full", "Logistic Regression baseline", duo_log_accuracy, duo_log_precision, duo_log_recall, duo_log_f1, duo_log_roc, duo_log_pr))

print("Duo Logistic Regression classification")
print("Accuracy:", duo_log_accuracy)
print("Precision:", duo_log_precision)
print("Recall:", duo_log_recall)
print("F1:", duo_log_f1)
print("ROC-AUC:", duo_log_roc)
print("PR-AUC:", duo_log_pr)
duo_log_pred.groupBy("isTop25", "prediction").count().orderBy("isTop25", "prediction").show()

In [ ]:
# Duo GBT Classifier with full team-composition features.
duo_cls_gbt_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
duo_cls_gbt = GBTClassifier(
    labelCol="isTop25",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

duo_cls_gbt_pipeline = Pipeline(stages=[duo_cls_gbt_assembler, duo_cls_gbt])
duo_cls_gbt_model = duo_cls_gbt_pipeline.fit(duoTrain)
duo_cls_gbt_pred = duo_cls_gbt_model.transform(duoTest)

duo_cls_gbt_accuracy = multiclass_eval_accuracy.evaluate(duo_cls_gbt_pred)
duo_cls_gbt_roc = binary_eval_roc.evaluate(duo_cls_gbt_pred)
duo_cls_gbt_pr = binary_eval_pr.evaluate(duo_cls_gbt_pred)

# Compute Top25 precision, recall, and positive-class F1 from one small aggregate job.
# This avoids Spark's weighted-F1 definition and keeps the metric aligned with sklearn f1_score(pos_label=1).
duo_cls_gbt_confusion_counts = duo_cls_gbt_pred.agg(
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
    f.sum(f.when((f.col("isTop25") == 0) & (f.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 0), 1).otherwise(0)).alias("fn")
).collect()[0]

duo_cls_gbt_tp = duo_cls_gbt_confusion_counts["tp"]
duo_cls_gbt_fp = duo_cls_gbt_confusion_counts["fp"]
duo_cls_gbt_fn = duo_cls_gbt_confusion_counts["fn"]

duo_cls_gbt_precision = duo_cls_gbt_tp / (duo_cls_gbt_tp + duo_cls_gbt_fp) if (duo_cls_gbt_tp + duo_cls_gbt_fp) > 0 else 0
duo_cls_gbt_recall = duo_cls_gbt_tp / (duo_cls_gbt_tp + duo_cls_gbt_fn) if (duo_cls_gbt_tp + duo_cls_gbt_fn) > 0 else 0
duo_cls_gbt_f1 = 2 * duo_cls_gbt_precision * duo_cls_gbt_recall / (duo_cls_gbt_precision + duo_cls_gbt_recall) if (duo_cls_gbt_precision + duo_cls_gbt_recall) > 0 else 0

classification_results.append(("duo", "team_full", "GBTClassifier", duo_cls_gbt_accuracy, duo_cls_gbt_precision, duo_cls_gbt_recall, duo_cls_gbt_f1, duo_cls_gbt_roc, duo_cls_gbt_pr))

print("Duo GBT classification")
print("Accuracy:", duo_cls_gbt_accuracy)
print("Precision:", duo_cls_gbt_precision)
print("Recall:", duo_cls_gbt_recall)
print("F1:", duo_cls_gbt_f1)
print("ROC-AUC:", duo_cls_gbt_roc)
print("PR-AUC:", duo_cls_gbt_pr)
duo_cls_gbt_pred.groupBy("isTop25", "prediction").count().orderBy("isTop25", "prediction").show()

## 12.3 Squad classification

For squad classification, we again use the full team-composition feature set.

In [ ]:
# Squad Logistic Regression with full team-composition features.
squad_log_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
squad_log_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
squad_log = LogisticRegression(labelCol="isTop25", featuresCol="scaledFeatures", maxIter=50)

squad_log_pipeline = Pipeline(stages=[squad_log_assembler, squad_log_scaler, squad_log])
squad_log_model = squad_log_pipeline.fit(squadTrain)
squad_log_pred = squad_log_model.transform(squadTest)

squad_log_accuracy = multiclass_eval_accuracy.evaluate(squad_log_pred)
squad_log_roc = binary_eval_roc.evaluate(squad_log_pred)
squad_log_pr = binary_eval_pr.evaluate(squad_log_pred)

# Compute Top25 precision, recall, and positive-class F1 from one small aggregate job.
# This avoids Spark's weighted-F1 definition and keeps the metric aligned with sklearn f1_score(pos_label=1).
squad_log_confusion_counts = squad_log_pred.agg(
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
    f.sum(f.when((f.col("isTop25") == 0) & (f.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 0), 1).otherwise(0)).alias("fn")
).collect()[0]

squad_log_tp = squad_log_confusion_counts["tp"]
squad_log_fp = squad_log_confusion_counts["fp"]
squad_log_fn = squad_log_confusion_counts["fn"]

squad_log_precision = squad_log_tp / (squad_log_tp + squad_log_fp) if (squad_log_tp + squad_log_fp) > 0 else 0
squad_log_recall = squad_log_tp / (squad_log_tp + squad_log_fn) if (squad_log_tp + squad_log_fn) > 0 else 0
squad_log_f1 = 2 * squad_log_precision * squad_log_recall / (squad_log_precision + squad_log_recall) if (squad_log_precision + squad_log_recall) > 0 else 0

classification_results.append(("squad", "team_full", "Logistic Regression baseline", squad_log_accuracy, squad_log_precision, squad_log_recall, squad_log_f1, squad_log_roc, squad_log_pr))

print("Squad Logistic Regression classification")
print("Accuracy:", squad_log_accuracy)
print("Precision:", squad_log_precision)
print("Recall:", squad_log_recall)
print("F1:", squad_log_f1)
print("ROC-AUC:", squad_log_roc)
print("PR-AUC:", squad_log_pr)
squad_log_pred.groupBy("isTop25", "prediction").count().orderBy("isTop25", "prediction").show()

In [ ]:
# Squad GBT Classifier with full team-composition features.
squad_cls_gbt_assembler = VectorAssembler(inputCols=team_full_features, outputCol="features", handleInvalid="keep")
squad_cls_gbt = GBTClassifier(
    labelCol="isTop25",
    featuresCol="features",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    seed=RANDOM_STATE
)

squad_cls_gbt_pipeline = Pipeline(stages=[squad_cls_gbt_assembler, squad_cls_gbt])
squad_cls_gbt_model = squad_cls_gbt_pipeline.fit(squadTrain)
squad_cls_gbt_pred = squad_cls_gbt_model.transform(squadTest)

squad_cls_gbt_accuracy = multiclass_eval_accuracy.evaluate(squad_cls_gbt_pred)
squad_cls_gbt_roc = binary_eval_roc.evaluate(squad_cls_gbt_pred)
squad_cls_gbt_pr = binary_eval_pr.evaluate(squad_cls_gbt_pred)

# Compute Top25 precision, recall, and positive-class F1 from one small aggregate job.
# This avoids Spark's weighted-F1 definition and keeps the metric aligned with sklearn f1_score(pos_label=1).
squad_cls_gbt_confusion_counts = squad_cls_gbt_pred.agg(
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
    f.sum(f.when((f.col("isTop25") == 0) & (f.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
    f.sum(f.when((f.col("isTop25") == 1) & (f.col("prediction") == 0), 1).otherwise(0)).alias("fn")
).collect()[0]

squad_cls_gbt_tp = squad_cls_gbt_confusion_counts["tp"]
squad_cls_gbt_fp = squad_cls_gbt_confusion_counts["fp"]
squad_cls_gbt_fn = squad_cls_gbt_confusion_counts["fn"]

squad_cls_gbt_precision = squad_cls_gbt_tp / (squad_cls_gbt_tp + squad_cls_gbt_fp) if (squad_cls_gbt_tp + squad_cls_gbt_fp) > 0 else 0
squad_cls_gbt_recall = squad_cls_gbt_tp / (squad_cls_gbt_tp + squad_cls_gbt_fn) if (squad_cls_gbt_tp + squad_cls_gbt_fn) > 0 else 0
squad_cls_gbt_f1 = 2 * squad_cls_gbt_precision * squad_cls_gbt_recall / (squad_cls_gbt_precision + squad_cls_gbt_recall) if (squad_cls_gbt_precision + squad_cls_gbt_recall) > 0 else 0

classification_results.append(("squad", "team_full", "GBTClassifier", squad_cls_gbt_accuracy, squad_cls_gbt_precision, squad_cls_gbt_recall, squad_cls_gbt_f1, squad_cls_gbt_roc, squad_cls_gbt_pr))

print("Squad GBT classification")
print("Accuracy:", squad_cls_gbt_accuracy)
print("Precision:", squad_cls_gbt_precision)
print("Recall:", squad_cls_gbt_recall)
print("F1:", squad_cls_gbt_f1)
print("ROC-AUC:", squad_cls_gbt_roc)
print("PR-AUC:", squad_cls_gbt_pr)
squad_cls_gbt_pred.groupBy("isTop25", "prediction").count().orderBy("isTop25", "prediction").show()

## 12.4 Classification result table

In [ ]:
# The classification metrics are stored in a small Python list.
# Use pandas for the final report table to avoid creating a Spark DataFrame from local objects.

classificationResults = pd.DataFrame(
    classification_results,
    columns=["mode", "unit_or_version", "model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
)

classificationResults = classificationResults.sort_values(["mode", "unit_or_version", "model"])
print(classificationResults.to_string(index=False))
classificationResults.to_csv(os.path.join(OUTPUT_DIR, "classification_results.csv"), index=False)


## 12.5 Classification feature importance

For the Top 25% classification task, we also export feature importance from the GBT classifiers.

These tables help compare whether the variables that explain exact placement also help separate Top 25% players or teams from the rest.

In [ ]:
# GBT classifier feature importance tables.
# These are read from the trained Spark MLlib model objects and converted to small pandas tables.
# No Spark DataFrame is created from local Python lists.

classification_importance_tables = []

solo_cls_gbt_importance_values = solo_cls_gbt_model.stages[-1].featureImportances.toArray().tolist()
solo_cls_importance_rows = []
for i in range(len(solo_features)):
    solo_cls_importance_rows.append(("solo", "player", solo_features[i], float(solo_cls_gbt_importance_values[i])))

soloClsGbtImportance = pd.DataFrame(solo_cls_importance_rows, columns=["mode", "unit_or_version", "feature", "importance"])
soloClsGbtImportance = soloClsGbtImportance.sort_values("importance", ascending=False)
print("Solo GBT classification feature importance")
print(soloClsGbtImportance.head(20).to_string(index=False))
soloClsGbtImportance.to_csv(os.path.join(OUTPUT_DIR, "solo_gbt_classification_feature_importance.csv"), index=False)
classification_importance_tables.append(soloClsGbtImportance)

duo_cls_gbt_importance_values = duo_cls_gbt_model.stages[-1].featureImportances.toArray().tolist()
duo_cls_importance_rows = []
for i in range(len(team_full_features)):
    duo_cls_importance_rows.append(("duo", "team_full", team_full_features[i], float(duo_cls_gbt_importance_values[i])))

duoClsGbtImportance = pd.DataFrame(duo_cls_importance_rows, columns=["mode", "unit_or_version", "feature", "importance"])
duoClsGbtImportance = duoClsGbtImportance.sort_values("importance", ascending=False)
print("Duo GBT classification feature importance")
print(duoClsGbtImportance.head(20).to_string(index=False))
duoClsGbtImportance.to_csv(os.path.join(OUTPUT_DIR, "duo_gbt_classification_feature_importance.csv"), index=False)
classification_importance_tables.append(duoClsGbtImportance)

squad_cls_gbt_importance_values = squad_cls_gbt_model.stages[-1].featureImportances.toArray().tolist()
squad_cls_importance_rows = []
for i in range(len(team_full_features)):
    squad_cls_importance_rows.append(("squad", "team_full", team_full_features[i], float(squad_cls_gbt_importance_values[i])))

squadClsGbtImportance = pd.DataFrame(squad_cls_importance_rows, columns=["mode", "unit_or_version", "feature", "importance"])
squadClsGbtImportance = squadClsGbtImportance.sort_values("importance", ascending=False)
print("Squad GBT classification feature importance")
print(squadClsGbtImportance.head(20).to_string(index=False))
squadClsGbtImportance.to_csv(os.path.join(OUTPUT_DIR, "squad_gbt_classification_feature_importance.csv"), index=False)
classification_importance_tables.append(squadClsGbtImportance)

classificationGbtImportanceAll = pd.concat(classification_importance_tables, ignore_index=True)
classificationGbtImportanceAll.to_csv(os.path.join(OUTPUT_DIR, "classification_gbt_feature_importance_all_modes.csv"), index=False)


# 13. Clustering: player styles and team styles

Clustering answers a different question:

> Can we discover different PUBG playstyles or team styles from behavior variables?

Important design decisions:

- `winPlacePerc` is **not** used to form clusters.
- After clusters are created, we compare each cluster's average `winPlacePerc` and Top 25% rate.
- For duo and squad, clustering uses **full teams only** so that K-Means does not simply separate teams by team size.


In [ ]:
solo_cluster_features = [
    "kills", "damageDealt", "DBNOs", "assists", "revives", "boosts", "heals",
    "weaponsAcquired", "walkDistance", "rideDistance", "swimDistance", "totalDistance",
    "damagePerKill", "headshotRate", "healsAndBoosts", "boostHealRatio",
    "supportActions", "combatContribution"
]

team_cluster_features = [
    "kills_mean", "damageDealt_mean", "DBNOs_mean", "assists_mean", "revives_mean",
    "boosts_mean", "heals_mean", "weaponsAcquired_mean", "walkDistance_mean",
    "rideDistance_mean", "swimDistance_mean", "totalDistance_mean", "damagePerKill_mean",
    "headshotRate_mean", "healsAndBoosts_mean", "boostHealRatio_mean",
    "supportActions_mean", "combatContribution_mean", "kills_max", "damageDealt_max",
    "boosts_max", "walkDistance_max", "kills_std", "damageDealt_std", "boosts_std",
    "walkDistance_std", "topKillShare", "topDamageShare", "topBoostShare", "topWalkShare"
]

# Full teams only for duo/squad clustering.
duoClusterData = duoTeamData.filter(f.col("isFullTeam") == 1)
squadClusterData = squadTeamData.filter(f.col("isFullTeam") == 1)


## 13.1 K selection with silhouette score

We inspect K from 3 to 8.  
For speed, K selection uses a sample. The final K-Means model is then fitted on the full clustering dataset for each mode.

In [ ]:
cluster_eval = ClusteringEvaluator(featuresCol="scaledFeatures", predictionCol="prediction", metricName="silhouette")

# Solo K selection.
# The KMeans fitting and silhouette evaluation are still Spark MLlib operations.
# Only the final small score table is stored with pandas.

soloKRows = []
soloClusterSample = soloData.sample(False, 0.20, seed=RANDOM_STATE)

for k in range(3, 9):
    solo_k_assembler = VectorAssembler(inputCols=solo_cluster_features, outputCol="features", handleInvalid="keep")
    solo_k_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
    solo_kmeans = KMeans(k=k, seed=RANDOM_STATE, featuresCol="scaledFeatures", predictionCol="prediction")
    solo_k_pipeline = Pipeline(stages=[solo_k_assembler, solo_k_scaler, solo_kmeans])
    solo_k_model = solo_k_pipeline.fit(soloClusterSample)
    solo_k_pred = solo_k_model.transform(soloClusterSample)
    solo_k_silhouette = cluster_eval.evaluate(solo_k_pred)
    soloKRows.append(("solo", k, solo_k_silhouette))
    print("solo k =", k, "silhouette =", solo_k_silhouette)

soloKResult = pd.DataFrame(soloKRows, columns=["mode", "k", "silhouette"])
print(soloKResult.to_string(index=False))


In [ ]:
# Duo K selection.
# KMeans remains Spark MLlib; only the final local score table uses pandas.

duoKRows = []
duoClusterSample = duoClusterData.sample(False, 0.30, seed=RANDOM_STATE)

for k in range(3, 9):
    duo_k_assembler = VectorAssembler(inputCols=team_cluster_features, outputCol="features", handleInvalid="keep")
    duo_k_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
    duo_kmeans = KMeans(k=k, seed=RANDOM_STATE, featuresCol="scaledFeatures", predictionCol="prediction")
    duo_k_pipeline = Pipeline(stages=[duo_k_assembler, duo_k_scaler, duo_kmeans])
    duo_k_model = duo_k_pipeline.fit(duoClusterSample)
    duo_k_pred = duo_k_model.transform(duoClusterSample)
    duo_k_silhouette = cluster_eval.evaluate(duo_k_pred)
    duoKRows.append(("duo", k, duo_k_silhouette))
    print("duo k =", k, "silhouette =", duo_k_silhouette)

duoKResult = pd.DataFrame(duoKRows, columns=["mode", "k", "silhouette"])
print(duoKResult.to_string(index=False))


In [ ]:
# Squad K selection.
# KMeans remains Spark MLlib; only the final local score table uses pandas.

squadKRows = []
squadClusterSample = squadClusterData.sample(False, 0.30, seed=RANDOM_STATE)

for k in range(3, 9):
    squad_k_assembler = VectorAssembler(inputCols=team_cluster_features, outputCol="features", handleInvalid="keep")
    squad_k_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
    squad_kmeans = KMeans(k=k, seed=RANDOM_STATE, featuresCol="scaledFeatures", predictionCol="prediction")
    squad_k_pipeline = Pipeline(stages=[squad_k_assembler, squad_k_scaler, squad_kmeans])
    squad_k_model = squad_k_pipeline.fit(squadClusterSample)
    squad_k_pred = squad_k_model.transform(squadClusterSample)
    squad_k_silhouette = cluster_eval.evaluate(squad_k_pred)
    squadKRows.append(("squad", k, squad_k_silhouette))
    print("squad k =", k, "silhouette =", squad_k_silhouette)

squadKResult = pd.DataFrame(squadKRows, columns=["mode", "k", "silhouette"])
print(squadKResult.to_string(index=False))


In [ ]:
# Combine K-selection results.
# These are small local result tables, so pandas avoids unnecessary Spark jobs.

kSelectionResult = pd.concat([soloKResult, duoKResult, squadKResult], ignore_index=True)
print(kSelectionResult.to_string(index=False))
kSelectionResult.to_csv(os.path.join(OUTPUT_DIR, "kmeans_silhouette_scores.csv"), index=False)


## 13.2 Fit final K-Means models

The default final K is 5 for each mode because 4–5 clusters are easier to interpret in a report and presentation.  
If your silhouette table strongly suggests another K, change `SOLO_K`, `DUO_K`, or `SQUAD_K` in the configuration section.

In [ ]:
# Final solo K-Means.
solo_cluster_assembler = VectorAssembler(inputCols=solo_cluster_features, outputCol="features", handleInvalid="keep")
solo_cluster_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
solo_kmeans = KMeans(k=SOLO_K, seed=RANDOM_STATE, featuresCol="scaledFeatures", predictionCol="cluster")

solo_cluster_pipeline = Pipeline(stages=[solo_cluster_assembler, solo_cluster_scaler, solo_kmeans])
solo_cluster_model = solo_cluster_pipeline.fit(soloData)
soloClustered = solo_cluster_model.transform(soloData)

soloClustered.groupBy("cluster").count().orderBy("cluster").show()

In [ ]:
# Final duo K-Means on full duo teams only.
duo_cluster_assembler = VectorAssembler(inputCols=team_cluster_features, outputCol="features", handleInvalid="keep")
duo_cluster_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
duo_kmeans = KMeans(k=DUO_K, seed=RANDOM_STATE, featuresCol="scaledFeatures", predictionCol="cluster")

duo_cluster_pipeline = Pipeline(stages=[duo_cluster_assembler, duo_cluster_scaler, duo_kmeans])
duo_cluster_model = duo_cluster_pipeline.fit(duoClusterData)
duoClustered = duo_cluster_model.transform(duoClusterData)

duoClustered.groupBy("cluster").count().orderBy("cluster").show()

In [ ]:
# Final squad K-Means on full squad teams only.
squad_cluster_assembler = VectorAssembler(inputCols=team_cluster_features, outputCol="features", handleInvalid="keep")
squad_cluster_scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
squad_kmeans = KMeans(k=SQUAD_K, seed=RANDOM_STATE, featuresCol="scaledFeatures", predictionCol="cluster")

squad_cluster_pipeline = Pipeline(stages=[squad_cluster_assembler, squad_cluster_scaler, squad_kmeans])
squad_cluster_model = squad_cluster_pipeline.fit(squadClusterData)
squadClustered = squad_cluster_model.transform(squadClusterData)

squadClustered.groupBy("cluster").count().orderBy("cluster").show()

## 13.3 Cluster profiles

The following tables are the main clustering outputs.

To name clusters, compare each cluster's:

- average `winPlacePerc`;
- Top 25% rate;
- average combat, movement, resource, and support features.

In [ ]:
# Solo cluster profile.
soloClusterProfile = (
    soloClustered
    .groupBy("cluster")
    .agg(
        f.count("*").alias("n_rows"),
        f.avg("winPlacePerc").alias("avg_winPlacePerc"),
        f.avg("isTop25").alias("top25_rate"),
        f.avg("kills").alias("avg_kills"),
        f.avg("damageDealt").alias("avg_damageDealt"),
        f.avg("boosts").alias("avg_boosts"),
        f.avg("heals").alias("avg_heals"),
        f.avg("walkDistance").alias("avg_walkDistance"),
        f.avg("rideDistance").alias("avg_rideDistance"),
        f.avg("totalDistance").alias("avg_totalDistance"),
        f.avg("weaponsAcquired").alias("avg_weaponsAcquired"),
        f.avg("supportActions").alias("avg_supportActions"),
        f.avg("combatContribution").alias("avg_combatContribution")
    )
    .orderBy(f.desc("avg_winPlacePerc"))
)

soloClusterProfile.show(truncate=False)
soloClusterProfile.toPandas().to_csv(os.path.join(OUTPUT_DIR, "solo_cluster_profile.csv"), index=False)

In [ ]:
# Duo cluster profile.
duoClusterProfile = (
    duoClustered
    .groupBy("cluster")
    .agg(
        f.count("*").alias("n_teams"),
        f.avg("winPlacePerc").alias("avg_winPlacePerc"),
        f.avg("isTop25").alias("top25_rate"),
        f.avg("kills_sum").alias("avg_kills_sum"),
        f.avg("damageDealt_sum").alias("avg_damageDealt_sum"),
        f.avg("DBNOs_sum").alias("avg_DBNOs_sum"),
        f.avg("assists_sum").alias("avg_assists_sum"),
        f.avg("revives_sum").alias("avg_revives_sum"),
        f.avg("boosts_sum").alias("avg_boosts_sum"),
        f.avg("heals_sum").alias("avg_heals_sum"),
        f.avg("walkDistance_sum").alias("avg_walkDistance_sum"),
        f.avg("rideDistance_sum").alias("avg_rideDistance_sum"),
        f.avg("totalDistance_sum").alias("avg_totalDistance_sum"),
        f.avg("kills_mean").alias("avg_kills_mean"),
        f.avg("damageDealt_mean").alias("avg_damageDealt_mean"),
        f.avg("boosts_mean").alias("avg_boosts_mean"),
        f.avg("topKillShare").alias("avg_topKillShare"),
        f.avg("topDamageShare").alias("avg_topDamageShare")
    )
    .orderBy(f.desc("avg_winPlacePerc"))
)

duoClusterProfile.show(truncate=False)
duoClusterProfile.toPandas().to_csv(os.path.join(OUTPUT_DIR, "duo_cluster_profile.csv"), index=False)

In [ ]:
# Squad cluster profile.
squadClusterProfile = (
    squadClustered
    .groupBy("cluster")
    .agg(
        f.count("*").alias("n_teams"),
        f.avg("winPlacePerc").alias("avg_winPlacePerc"),
        f.avg("isTop25").alias("top25_rate"),
        f.avg("kills_sum").alias("avg_kills_sum"),
        f.avg("damageDealt_sum").alias("avg_damageDealt_sum"),
        f.avg("DBNOs_sum").alias("avg_DBNOs_sum"),
        f.avg("assists_sum").alias("avg_assists_sum"),
        f.avg("revives_sum").alias("avg_revives_sum"),
        f.avg("boosts_sum").alias("avg_boosts_sum"),
        f.avg("heals_sum").alias("avg_heals_sum"),
        f.avg("walkDistance_sum").alias("avg_walkDistance_sum"),
        f.avg("rideDistance_sum").alias("avg_rideDistance_sum"),
        f.avg("totalDistance_sum").alias("avg_totalDistance_sum"),
        f.avg("kills_mean").alias("avg_kills_mean"),
        f.avg("damageDealt_mean").alias("avg_damageDealt_mean"),
        f.avg("boosts_mean").alias("avg_boosts_mean"),
        f.avg("topKillShare").alias("avg_topKillShare"),
        f.avg("topDamageShare").alias("avg_topDamageShare")
    )
    .orderBy(f.desc("avg_winPlacePerc"))
)

squadClusterProfile.show(truncate=False)
squadClusterProfile.toPandas().to_csv(os.path.join(OUTPUT_DIR, "squad_cluster_profile.csv"), index=False)

## 13.4 Cluster distinguishing features

The cluster profile tables show the average value of selected variables.

This additional table compares every cluster's mean feature values against the mode-wide mean using z-scores. It is used to name clusters more objectively.

- Positive z-score: this cluster is above the mode average for that feature.
- Negative z-score: this cluster is below the mode average for that feature.

In [ ]:
# Distinguishing features for cluster naming.
# The final explanation tables are small pandas tables.
# The Spark work here is limited to group-level averages and overall averages/stddevs.

cluster_distinguishing_all = []

cluster_sources = [
    ("solo", soloClustered, solo_cluster_features),
    ("duo", duoClustered, team_cluster_features),
    ("squad", squadClustered, team_cluster_features)
]

for mode_name, clustered_df, feature_list in cluster_sources:
    print("Distinguishing features for", mode_name)

    mean_exprs = []
    std_exprs = []
    cluster_mean_exprs = []

    for c in feature_list:
        mean_exprs.append(f.avg(c).alias(c))
        std_exprs.append(f.stddev_samp(c).alias(c))
        cluster_mean_exprs.append(f.avg(c).alias(c))

    overall_mean_row = clustered_df.agg(*mean_exprs).collect()[0].asDict()
    overall_std_row = clustered_df.agg(*std_exprs).collect()[0].asDict()

    cluster_mean_rows = (
        clustered_df
        .groupBy("cluster")
        .agg(*cluster_mean_exprs)
        .orderBy("cluster")
        .collect()
    )

    explanation_rows = []

    for row in cluster_mean_rows:
        row_dict = row.asDict()
        cluster_id = row_dict["cluster"]
        z_values = []

        for c in feature_list:
            feature_mean = row_dict[c]
            overall_mean = overall_mean_row[c]
            overall_std = overall_std_row[c]

            if overall_std is not None and overall_std != 0:
                z_score = (feature_mean - overall_mean) / overall_std
                z_values.append((c, z_score))

        z_values_sorted = sorted(z_values, key=lambda x: x[1], reverse=True)
        top_above = z_values_sorted[:6]
        top_below = z_values_sorted[-6:]

        explanation_rows.append({
            "mode": mode_name,
            "cluster": cluster_id,
            "top_above_average_features": ", ".join([name + " (" + format(value, ".2f") + ")" for name, value in top_above]),
            "top_below_average_features": ", ".join([name + " (" + format(value, ".2f") + ")" for name, value in top_below])
        })

    explanation_df = pd.DataFrame(explanation_rows)
    print(explanation_df.to_string(index=False))
    explanation_df.to_csv(os.path.join(OUTPUT_DIR, "cluster_distinguishing_features_" + mode_name + ".csv"), index=False)
    cluster_distinguishing_all.append(explanation_df)

clusterDistinguishingAll = pd.concat(cluster_distinguishing_all, ignore_index=True)
clusterDistinguishingAll.to_csv(os.path.join(OUTPUT_DIR, "cluster_distinguishing_features_all_modes.csv"), index=False)


## 13.4 PCA visualization for clusters

PCA is only for visualization.  
K-Means itself was fitted in the full standardized feature space.

To keep the plot readable, this section samples rows before converting to pandas.

In [ ]:
# Solo PCA plot.
soloPcaSample = soloClustered.sample(False, 0.05, seed=RANDOM_STATE)
solo_pca = PCA(k=2, inputCol="scaledFeatures", outputCol="pcaFeatures")
solo_pca_model = solo_pca.fit(soloPcaSample)
solo_pca_result = solo_pca_model.transform(soloPcaSample).select("pcaFeatures", "cluster").toPandas()

solo_pca_result["pc1"] = solo_pca_result["pcaFeatures"].apply(lambda x: float(x[0]))
solo_pca_result["pc2"] = solo_pca_result["pcaFeatures"].apply(lambda x: float(x[1]))

plt.figure(figsize=(8, 6))
plt.scatter(solo_pca_result["pc1"], solo_pca_result["pc2"], c=solo_pca_result["cluster"], s=8, alpha=0.6)
plt.title("Solo player clusters: PCA visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

In [ ]:
# Duo PCA plot.
duoPcaSample = duoClustered.sample(False, 0.10, seed=RANDOM_STATE)
duo_pca = PCA(k=2, inputCol="scaledFeatures", outputCol="pcaFeatures")
duo_pca_model = duo_pca.fit(duoPcaSample)
duo_pca_result = duo_pca_model.transform(duoPcaSample).select("pcaFeatures", "cluster").toPandas()

duo_pca_result["pc1"] = duo_pca_result["pcaFeatures"].apply(lambda x: float(x[0]))
duo_pca_result["pc2"] = duo_pca_result["pcaFeatures"].apply(lambda x: float(x[1]))

plt.figure(figsize=(8, 6))
plt.scatter(duo_pca_result["pc1"], duo_pca_result["pc2"], c=duo_pca_result["cluster"], s=8, alpha=0.6)
plt.title("Duo full-team clusters: PCA visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

In [ ]:
# Squad PCA plot.
squadPcaSample = squadClustered.sample(False, 0.10, seed=RANDOM_STATE)
squad_pca = PCA(k=2, inputCol="scaledFeatures", outputCol="pcaFeatures")
squad_pca_model = squad_pca.fit(squadPcaSample)
squad_pca_result = squad_pca_model.transform(squadPcaSample).select("pcaFeatures", "cluster").toPandas()

squad_pca_result["pc1"] = squad_pca_result["pcaFeatures"].apply(lambda x: float(x[0]))
squad_pca_result["pc2"] = squad_pca_result["pcaFeatures"].apply(lambda x: float(x[1]))

plt.figure(figsize=(8, 6))
plt.scatter(squad_pca_result["pc1"], squad_pca_result["pc2"], c=squad_pca_result["cluster"], s=8, alpha=0.6)
plt.title("Squad full-team clusters: PCA visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

---
# 14. Focused analysis: boosts vs heals

EDA suggested that boosts are more consistently associated with high placement than heals.

This section creates a compact table for each mode.

Definitions:

- `some_heals`: heals greater than the median;
- `high_boosts`: boosts greater than the 95th percentile.

For solo, heals/boosts are player-level.  
For duo/squad, heals/boosts are team-level sums.

In [ ]:
# Solo boosts vs heals.
solo_heal_median = soloData.approxQuantile("heals", [0.5], 0.001)[0]
solo_boost_p95 = soloData.approxQuantile("boosts", [0.95], 0.001)[0]

soloBoostHeal = (
    soloData
    .withColumn("heal_group", f.when(f.col("heals") > solo_heal_median, f.lit("some_heals")).otherwise(f.lit("low_or_no_heals")))
    .withColumn("boost_group", f.when(f.col("boosts") > solo_boost_p95, f.lit("high_boosts")).otherwise(f.lit("not_high_boosts")))
    .groupBy("heal_group", "boost_group")
    .agg(
        f.count("*").alias("n_rows"),
        f.avg("winPlacePerc").alias("avg_winPlacePerc"),
        f.avg("isTop25").alias("top25_rate"),
        f.avg("heals").alias("avg_heals"),
        f.avg("boosts").alias("avg_boosts")
    )
    .orderBy(f.desc("avg_winPlacePerc"))
)

soloBoostHeal.show(truncate=False)
soloBoostHeal.toPandas().to_csv(os.path.join(OUTPUT_DIR, "solo_boosts_heals_summary.csv"), index=False)

In [ ]:
# Duo boosts vs heals.
duo_heal_median = duoTeamData.approxQuantile("heals_sum", [0.5], 0.001)[0]
duo_boost_p95 = duoTeamData.approxQuantile("boosts_sum", [0.95], 0.001)[0]

duoBoostHeal = (
    duoTeamData
    .withColumn("heal_group", f.when(f.col("heals_sum") > duo_heal_median, f.lit("some_heals")).otherwise(f.lit("low_or_no_heals")))
    .withColumn("boost_group", f.when(f.col("boosts_sum") > duo_boost_p95, f.lit("high_boosts")).otherwise(f.lit("not_high_boosts")))
    .groupBy("heal_group", "boost_group")
    .agg(
        f.count("*").alias("n_teams"),
        f.avg("winPlacePerc").alias("avg_winPlacePerc"),
        f.avg("isTop25").alias("top25_rate"),
        f.avg("heals_sum").alias("avg_heals_sum"),
        f.avg("boosts_sum").alias("avg_boosts_sum")
    )
    .orderBy(f.desc("avg_winPlacePerc"))
)

duoBoostHeal.show(truncate=False)
duoBoostHeal.toPandas().to_csv(os.path.join(OUTPUT_DIR, "duo_boosts_heals_summary.csv"), index=False)

In [ ]:
# Squad boosts vs heals.
squad_heal_median = squadTeamData.approxQuantile("heals_sum", [0.5], 0.001)[0]
squad_boost_p95 = squadTeamData.approxQuantile("boosts_sum", [0.95], 0.001)[0]

squadBoostHeal = (
    squadTeamData
    .withColumn("heal_group", f.when(f.col("heals_sum") > squad_heal_median, f.lit("some_heals")).otherwise(f.lit("low_or_no_heals")))
    .withColumn("boost_group", f.when(f.col("boosts_sum") > squad_boost_p95, f.lit("high_boosts")).otherwise(f.lit("not_high_boosts")))
    .groupBy("heal_group", "boost_group")
    .agg(
        f.count("*").alias("n_teams"),
        f.avg("winPlacePerc").alias("avg_winPlacePerc"),
        f.avg("isTop25").alias("top25_rate"),
        f.avg("heals_sum").alias("avg_heals_sum"),
        f.avg("boosts_sum").alias("avg_boosts_sum")
    )
    .orderBy(f.desc("avg_winPlacePerc"))
)

squadBoostHeal.show(truncate=False)
squadBoostHeal.toPandas().to_csv(os.path.join(OUTPUT_DIR, "squad_boosts_heals_summary.csv"), index=False)